In [7]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import os

project_path = '/content/drive/MyDrive/Dist-Abl-PRL-All-Exs-ETTH1'
os.chdir(project_path)

print("Current Directory:", os.getcwd())

Current Directory: /content/drive/MyDrive/Dist-Abl-PRL-All-Exs-ETTH1


In [9]:
import os

# Set your Informer data directory
informer_path = f'{project_path}/Informer2020-original'
data_dir = f'{informer_path}/data/ETT'
os.makedirs(data_dir, exist_ok=True)

# File path
data_file = f'{data_dir}/ETTh1.csv'

# Download ETTh1 dataset
if not os.path.exists(data_file):
    print("Downloading ETTh1.csv...")
    os.system(
        f'wget -q "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv" '
        f'-O "{data_file}"'
    )

# Verify
if os.path.exists(data_file):
    print(f"✅ Download successful: {data_file}")
else:
    print("❌ Download failed.")

✅ Download successful: /content/drive/MyDrive/Dist-Abl-PRL-All-Exs-ETTH1/Informer2020-original/data/ETT/ETTh1.csv


In [10]:
# Fix np.Inf -> np.inf in tools.py (removed in NumPy 2.0)
tools_path = f'{project_path}/Informer2020-original/utils/tools.py'

with open(tools_path, 'r') as f:
    content = f.read()

content_fixed = content.replace('np.Inf', 'np.inf')

with open(tools_path, 'w') as f:
    f.write(content_fixed)

print("✅ Fixed np.Inf -> np.inf in utils/tools.py")

✅ Fixed np.Inf -> np.inf in utils/tools.py


In [11]:
# 2. Fix the import error
embed_file_path = '/content/drive/MyDrive/Dist-Abl-PRL-All-Exs-ETTH1/experiments/exp6_lod_post/models/embed.py'

with open(embed_file_path, 'r') as f:
    content = f.read()

content = content.replace(
    'from legendre_embedding import LegendrePositionEmbedding',
    'from models.legendre_embedding import LegendrePositionEmbedding'
)

with open(embed_file_path, 'w') as f:
    f.write(content)

print("✅ Import fixed!")


✅ Import fixed!


# Exp6 LOD Pre Phase 1 – Issues & Fixes

While conducting exp6 lod pre phase 1 through script file we faced 3 blockers

## 1. Module not found

**Legendre embedding**
We change the import to:

```
models.lengendre_embedding
```

## 2. --alpha variable not found(we replaced main_informer.py)

**Fix:**
We changed it to:

```
--decay_a
```

## 3. Matrix shape multiplication error

**Fix:**
We made a change in `encoder.py`

---

### Old Code

```python
def forward(self, x, attn_mask=None, delta_x=None):
        # x: combined embedding for Q/K
        # delta_x: clean delta for V
        
        attns = []
        if self.conv_layers is not None:
            for attn_layer, conv_layer in zip(self.attn_layers, self.conv_layers):
                # Pass delta_x through all encoder layers
                x, attn = attn_layer(x, attn_mask=attn_mask, delta_x=delta_x)
                x = conv_layer(x)
                attns.append(attn)
            x, attn = self.attn_layers[-1](x, attn_mask=attn_mask, delta_x=delta_x)
            attns.append(attn)
        else:
            for attn_layer in self.attn_layers:
                # Pass delta_x through all encoder layers
                x, attn = attn_layer(x, attn_mask=attn_mask, delta_x=delta_x)
                attns.append(attn)

        if self.norm is not None:
            x = self.norm(x)

        return x, attns
```

---

### New Code

```python
def forward(self, x, attn_mask=None, delta_x=None):
    attns = []
    if self.conv_layers is not None:
        for attn_layer, conv_layer in zip(self.attn_layers, self.conv_layers):
            x, attn = attn_layer(x, attn_mask=attn_mask, delta_x=delta_x)
            x = conv_layer(x)
            # ✅ FIX: slice delta_x to match downsampled x after each conv
            if delta_x is not None:
                delta_x = delta_x[:, :x.shape[1], :]
            attns.append(attn)
        x, attn = self.attn_layers[-1](x, attn_mask=attn_mask, delta_x=delta_x)
        attns.append(attn)
    else:
        for attn_layer in self.attn_layers:
            x, attn = attn_layer(x, attn_mask=attn_mask, delta_x=delta_x)
            attns.append(attn)

    if self.norm is not None:
        x = self.norm(x)

    return x, attns
```


In [32]:
!bash /content/drive/MyDrive/Dist-Abl-PRL-All-Exs-ETTH1/experiments/exp6_lod_post/exp6_lod_post_phase1.sh


Experiment 6-LOD-Pre — Phase 1: Label + Order + Distance (PRE-softmax)
Formula   : X'_i = X_i + T_i + P_i + O_i
  X_i  = value embedding (semantic)
  T_i  = temporal embedding
  P_i  = Legendre position label (legendre_embedding.py)
  O_i  = delta_x from embed.py — order in positional space
  w_ij = 1/(1+|i-j|^a) — baked into attn.py, PRE-softmax
         score_ij = w_ij * (Q_i · K_j)/sqrt(d), then softmax

Key distinctions:
  vs Exp6-LOD-Post : identical L+O+D but decay applied BEFORE softmax (not after)
  vs Exp5 (L+O)    : adds distance decay — isolates D's contribution
  vs Exp2-LOD      : O via delta_x/embed (not distance_operator), same decay pos
  vs Exp1-Pre      : adds L (Legendre) and O (delta_x) on top of decay

Alpha sweep : a ∈ {0.5, 1.0, 2.0}  (passed via --alpha)
  a=0.5 → slow decay | a=1.0 → linear | a=2.0 → fast decay
Seed       : 2021 | Pred lengths: 96, 192 | Total runs: 6
Start: Tue Apr 21 10:23:52 AM UTC 2026

Copying experiment files...
Files copied successfully.

###Key Features of Phase 2:

- Fixed alpha=1.0 (linear decay) - best performer from Phase 1 at both pred=96 (MSE=0.8694) and pred=192 (MSE=0.9571)
- 9 total runs: 3 seeds (2021, 2022, 2023) × 3 prediction lengths (48, 96, 192)
- Same architecture as Phase 1: L+O+D with pre-softmax distance decay

In [12]:
!bash /content/drive/MyDrive/Dist-Abl-PRL-All-Exs-ETTH1/experiments/exp6_lod_post/exp6_lod_post_phase2.sh


Experiment 6-LOD-Pre — Phase 2: Label + Order + Distance (PRE-softmax)
Formula   : X'_i = X_i + T_i + P_i + O_i
  X_i  = value embedding (semantic)
  T_i  = temporal embedding
  P_i  = Legendre position label (legendre_embedding.py)
  O_i  = delta_x from embed.py — order in positional space
  w_ij = 1/(1+|i-j|^1.0) — baked into attn.py, PRE-softmax
         score_ij = w_ij * (Q_i · K_j)/sqrt(d), then softmax

Phase 1 note: Alpha=1.0 won at BOTH pred=96 and pred=192
              pred=96  MSE=0.8694 (best of 3 alphas)
              pred=192 MSE=0.9571 (best of 3 alphas)

Alpha     : 1.0 (fixed — linear decay)
Seeds     : 2021, 2022, 2023
Pred lens : 48, 96, 192
Total runs: 9 (1 alpha × 3 seeds × 3 pred_lens)
Start: Tue Apr 21 01:14:01 PM UTC 2026

Copying experiment files...
Files copied successfully.

------------------------------------------------------------
RUN 1/9: exp6_lod_post_ph2_ETTh1_a1p0_pred48_seed2021
alpha=1.0 | pred_len=48 | seed=2021 | decay=pre-softmax (attn.py)
Start: